# Building a Deep Neural Network, Step by Step

**How this notebook works**
1. **Follow along (Parts 0–7).** We go through the notebook together. Run every cell and **read every output**: each function is tried on a small example before we use it.
2. **Exercises (at the end).** On your own. Each exercise has an empty cell; where the answer is deterministic, the expected output is shown so you can compare.
3. **Deliverable:** the completed notebook, with all exercises and answers, submitted at the end of the session. `Deadline: 21/10/2026: 11:59 PM` via my telegram individually.

**Notation, as in the lecture (examples are rows):**

| | shape |
|---|---|
| $\mathbf{A}^{[0]} = \mathbf{X}$ | $(m, n^{[0]})$ |
| $\mathbf{W}^{[l]}$, $d\mathbf{W}^{[l]}$ | $(n^{[l-1]}, n^{[l]})$ |
| $\mathbf{b}^{[l]}$, $d\mathbf{b}^{[l]}$ | $(1, n^{[l]})$ |
| $\mathbf{Z}^{[l]}$, $\mathbf{A}^{[l]}$, $d\mathbf{Z}^{[l]}$, $d\mathbf{A}^{[l]}$ | $(m, n^{[l]})$ |

If you look at the Deep Learning Specialization notebooks, they put examples in **columns** ($\mathbf{Z} = \mathbf{W}\mathbf{A} + \mathbf{b}$, $\mathbf{X}$ of shape $(n, m)$). Every formula here is the transpose of theirs.

## 0 · Setup

In [1]:
import torch
import matplotlib.pyplot as plt

torch.set_printoptions(precision=4, sci_mode=False)
print("PyTorch", torch.__version__)

PyTorch 2.15.0+cu132


## 1 · The plan

A deep network with $L$ layers repeats one block: a **linear** step followed by an **activation**. Hidden layers use ReLU; the output layer uses sigmoid, because we do binary classification.

```
 forward:   X ─▶ [LINEAR ─▶ RELU] × (L−1) ─▶ LINEAR ─▶ SIGMOID ─▶ AL ─▶ cost
                        │ cache                    │ cache
                        ▼                          ▼
 backward:  grads ◀─ [LINEAR ◀─ RELU] × (L−1) ◀─ LINEAR ◀─ SIGMOID ◀─ dZ[L] = AL − Y
```

During the forward pass each layer **stores a cache** (its input $\mathbf{A}^{[l-1]}$, its parameters, and $\mathbf{Z}^{[l]}$). The backward pass reads these caches in reverse order.

We write nine functions, each one a line or two of the lecture:

| Function | Computes |
|---|---|
| `initialize_parameters_deep` | $\mathbf{W}^{[l]}$, $\mathbf{b}^{[l]}$ for every layer |
| `linear_forward` | $\mathbf{Z}^{[l]} = \mathbf{A}^{[l-1]}\mathbf{W}^{[l]} + \mathbf{b}^{[l]}$ |
| `linear_activation_forward` | $\mathbf{A}^{[l]} = g^{[l]}(\mathbf{Z}^{[l]})$ |
| `L_model_forward` | the whole forward pass, $\mathbf{A}^{[L]}$ and the caches |
| `compute_cost` | $J = -\frac{1}{m}\sum_i \big(y_i \log a_i + (1-y_i)\log(1-a_i)\big)$ |
| `linear_backward` | $d\mathbf{W}^{[l]}$, $d\mathbf{b}^{[l]}$, $d\mathbf{A}^{[l-1]}$ from $d\mathbf{Z}^{[l]}$ |
| `linear_activation_backward` | $d\mathbf{Z}^{[l]} = d\mathbf{A}^{[l]} \odot g^{[l]\prime}(\mathbf{Z}^{[l]})$, then `linear_backward` |
| `L_model_backward` | the whole backward pass, all the gradients |
| `update_parameters` | $\mathbf{W}^{[l]} \leftarrow \mathbf{W}^{[l]} - \alpha\, d\mathbf{W}^{[l]}$, same for $\mathbf{b}^{[l]}$ |

## 2 · Initialization

`layer_dims` lists the sizes of **all** layers, from the input to the output: `[784, 20, 7, 5, 1]` is a network with 784 inputs, three hidden layers of 20, 7 and 5 units, and one output unit, so $L = 4$.

- $\mathbf{W}^{[l]}$: random, shape $(n^{[l-1]}, n^{[l]})$. Random values break the symmetry between units (Notebook 3, Part 2.5).
- $\mathbf{b}^{[l]}$: zeros, shape $(1, n^{[l]})$. Zeros are fine for $\mathbf{b}$, because the random $\mathbf{W}$ already breaks the symmetry.

**The scale.** For a 2-layer network, multiplying by 0.01 was enough. For deeper networks we multiply by $\sqrt{1/n^{[l-1]}}$, so that the values of $\mathbf{Z}$ keep a reasonable size from layer to layer. Exercise 4 shows what goes wrong otherwise; weeks 3–5 come back to it (Xavier and He initialization).

In [2]:
def initialize_parameters_deep(layer_dims, scale=None, seed=1):
    # layer_dims = [n0, n1, ..., nL]. W_l has shape (n_{l-1}, n_l), b_l has shape (1, n_l).
    # scale=None uses sqrt(1 / n_{l-1}); a number (e.g. 0.01) uses that number for every layer.
    torch.manual_seed(seed)
    params = {}
    for l in range(1, len(layer_dims)):
        n_prev, n_l = layer_dims[l - 1], layer_dims[l]
        s = scale if scale is not None else (1 / n_prev) ** 0.5
        params[f"W{l}"] = torch.randn(n_prev, n_l) * s
        params[f"b{l}"] = torch.zeros(1, n_l)
    return params

params = initialize_parameters_deep([784, 20, 7, 5, 1])
for name, p in params.items():
    print(f"{name}: {str(tuple(p.shape)):10s} std {p.std().item():.4f}" if name[0] == "W" else f"{name}: {tuple(p.shape)}")

W1: (784, 20)  std 0.0359
b1: (1, 20)
W2: (20, 7)    std 0.2253
b2: (1, 7)
W3: (7, 5)     std 0.3588
b3: (1, 5)
W4: (5, 1)     std 0.3239
b4: (1, 1)


In [3]:
params

{'W1': tensor([[-0.0545, -0.0268, -0.0234,  ..., -0.0415,  0.0250,  0.0071],
         [ 0.0309,  0.0087, -0.0237,  ...,  0.0016,  0.0055, -0.0170],
         [-0.0040,  0.0105, -0.0056,  ...,  0.0335, -0.0066, -0.0230],
         ...,
         [ 0.0464,  0.0022,  0.0276,  ..., -0.0449,  0.0670, -0.0283],
         [ 0.0162, -0.0174, -0.0795,  ..., -0.0562, -0.0284, -0.0362],
         [ 0.0054,  0.0143,  0.0213,  ..., -0.0249, -0.0149,  0.0033]]),
 'b1': tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]),
 'W2': tensor([[ 0.1041,  0.1509,  0.4660,  0.1008, -0.1269, -0.1568, -0.5315],
         [ 0.3737,  0.1972,  0.0166, -0.4295, -0.1059, -0.1259, -0.1081],
         [ 0.3099, -0.3019, -0.1240,  0.1553,  0.0289, -0.3259, -0.1611],
         [ 0.1156,  0.1939, -0.3795, -0.2357, -0.1177, -0.2039,  0.0519],
         [-0.1977,  0.0216,  0.0400, -0.2818,  0.0639,  0.2405,  0.3087],
         [-0.3243, -0.4158,  0.3859, -0.2035,  0.0627, -0.3123,  0.1781],
   

The standard deviation of $\mathbf{W}^{[1]}$ is about $\sqrt{1/784} \approx 0.036$, that of $\mathbf{W}^{[2]}$ about $\sqrt{1/20} \approx 0.22$, that of $\mathbf{W}^{[3]}$ about $\sqrt{1/7} \approx 0.38$. Each layer gets a scale adapted to the number of inputs it sums. ($\mathbf{W}^{[4]}$ has only 5 entries, too few for its standard deviation to be close to $\sqrt{1/5} \approx 0.45$.)

## 3 · Forward propagation

### 3.1 The linear step
$$\mathbf{Z}^{[l]} = \mathbf{A}^{[l-1]}\mathbf{W}^{[l]} + \mathbf{b}^{[l]}$$
$\mathbf{A}^{[l-1]}\mathbf{W}^{[l]}$ has shape $(m, n^{[l]})$; $\mathbf{b}^{[l]}$ has shape $(1, n^{[l]})$ and is **broadcast** to every row, that is, to every example.

In [4]:
def linear_forward(A_prev, W, b):
    # Z = A_prev W + b. The cache keeps what the backward pass will need.
    Z = A_prev @ W + b
    cache = (A_prev, W, b)
    return Z, cache

In [5]:
torch.manual_seed(0)
A_prev = torch.randn(3, 4)          # m = 3 examples, 4 features
W = torch.randn(4, 2)               # n[l-1] = 4, n[l] = 2
b = torch.tensor([[0.5, -0.5]])     # shape (1, 2)
Z, cache = linear_forward(A_prev, W, b)
print("Z =\n", Z)
print("shape of Z:", tuple(Z.shape))

Z =
 tensor([[ 0.2210, -0.5107],
        [ 1.9513, -2.9794],
        [ 1.7237, -1.8384]])
shape of Z: (3, 2)


In [6]:
cache

(tensor([[ 1.5410, -0.2934, -2.1788,  0.5684],
         [-1.0845, -1.3986,  0.4033,  0.8380],
         [-0.7193, -0.4033, -0.5966,  0.1820]]),
 tensor([[-0.8567,  1.1006],
         [-1.0712,  0.1227],
         [-0.5663,  0.3731],
         [-0.8920, -1.5091]]),
 tensor([[ 0.5000, -0.5000]]))

### The **cache** is used to store temporary intermediate values computed during the forward propagation pass so that they can be easily reused during the backward propagation pass.

### 3.2 The activation
`sigmoid` and `relu` return two things: the activation $\mathbf{A}$, and $\mathbf{Z}$ as a cache, because the derivative $g'(\mathbf{Z})$ is needed in the backward pass.

`linear_activation_forward` chains the two steps. Its cache is the pair `(linear_cache, activation_cache)`, that is, `((A_prev, W, b), Z)`.

In [7]:
def sigmoid(Z):
    return torch.sigmoid(Z), Z          # (A, cache); the cache is Z

def relu(Z):
    return torch.clamp(Z, min=0), Z     # max(0, Z), elementwise

def linear_activation_forward(A_prev, W, b, activation):
    Z, linear_cache = linear_forward(A_prev, W, b)
    if activation == "sigmoid":
        A, activation_cache = sigmoid(Z)
    elif activation == "relu":
        A, activation_cache = relu(Z)
    return A, (linear_cache, activation_cache)

for activation in ["relu", "sigmoid"]:
    A, cache = linear_activation_forward(A_prev, W, b, activation)
    print(f"{activation}:\n", A)

relu:
 tensor([[0.2210, 0.0000],
        [1.9513, 0.0000],
        [1.7237, 0.0000]])
sigmoid:
 tensor([[0.5550, 0.3750],
        [0.8756, 0.0484],
        [0.8486, 0.1372]])


ReLU replaces the negative values of $\mathbf{Z}$ by 0; sigmoid maps every value into $(0, 1)$.

### 3.3 The whole forward pass
`L_model_forward` loops over the $L - 1$ hidden layers with ReLU, then applies the output layer with sigmoid. It returns $\mathbf{A}^{[L]}$, one probability per example, and the list of the $L$ caches. The loop over **layers** is fine; inside a layer there is no loop over examples.

In [8]:
def L_model_forward(X, params):
    # [LINEAR -> RELU] x (L - 1), then LINEAR -> SIGMOID.
    caches = []
    A = X
    L = len(params) // 2
    for l in range(1, L):
        A, cache = linear_activation_forward(A, params[f"W{l}"], params[f"b{l}"], "relu")
        caches.append(cache)
    AL, cache = linear_activation_forward(A, params[f"W{L}"], params[f"b{L}"], "sigmoid")
    caches.append(cache)
    return AL, caches

torch.manual_seed(1)
X_small = torch.randn(6, 4)                         # m = 6 examples, 4 features
params_small = initialize_parameters_deep([4, 5, 3, 1])
AL, caches = L_model_forward(X_small, params_small)
print("AL =", AL.flatten())
print("shape of AL:", tuple(AL.shape), "| number of caches:", len(caches))
for l, ((A_in, W_l, b_l), Z_l) in enumerate(caches, start=1):
    print(f"layer {l}: A_prev {str(tuple(A_in.shape)):7s} W {str(tuple(W_l.shape)):7s} Z {tuple(Z_l.shape)}")

AL = tensor([0.6058, 0.5583, 0.4515, 0.5379, 0.4829, 0.5692])
shape of AL: (6, 1) | number of caches: 3
layer 1: A_prev (6, 4)  W (4, 5)  Z (6, 5)
layer 2: A_prev (6, 5)  W (5, 3)  Z (6, 3)
layer 3: A_prev (6, 3)  W (3, 1)  Z (6, 1)


## 4 · The cost

$$J = -\frac{1}{m}\sum_{i=1}^{m}\Big(y_i \log a^{[L]}_i + (1 - y_i)\log\big(1 - a^{[L]}_i\big)\Big)$$

`clamp` keeps $a$ inside $[10^{-7}, 1 - 10^{-7}]$, so that $\log 0$ never happens when the network is very confident.

**Worked example.** Three examples with $\mathbf{a} = (0.8, 0.9, 0.4)$ and $\mathbf{y} = (1, 1, 0)$:
$$J = -\tfrac{1}{3}\big(\log 0.8 + \log 0.9 + \log 0.6\big) = \tfrac{1}{3}(0.2231 + 0.1054 + 0.5108) = 0.2798$$

In [9]:
def compute_cost(AL, Y, eps=1e-7):
    # Binary cross-entropy, averaged over the m examples. AL and Y have shape (m, 1).
    AL = AL.clamp(eps, 1 - eps)        # avoids log(0)
    return -(Y * torch.log(AL) + (1 - Y) * torch.log(1 - AL)).mean()

AL_ex = torch.tensor([[0.8], [0.9], [0.4]])
Y_ex = torch.tensor([[1.0], [1.0], [0.0]])
print(f"cost = {compute_cost(AL_ex, Y_ex).item():.4f}")

cost = 0.2798


## 5 · Backward propagation

The four equations for layer $l$ (lecture slide "The equations for layer l"):

| | | shape |
|---|---|---|
| $d\mathbf{Z}^{[l]} = d\mathbf{A}^{[l]} \odot g^{[l]\prime}(\mathbf{Z}^{[l]})$ | elementwise product | $(m, n^{[l]})$ |
| $d\mathbf{W}^{[l]} = \frac{1}{m}\,\mathbf{A}^{[l-1]\top} d\mathbf{Z}^{[l]}$ | | $(n^{[l-1]}, n^{[l]})$ |
| $d\mathbf{b}^{[l]} = \frac{1}{m}\sum_{i} d\mathbf{Z}^{[l]}_{i,:}$ | sum over the examples (rows) | $(1, n^{[l]})$ |
| $d\mathbf{A}^{[l-1]} = d\mathbf{Z}^{[l]}\,\mathbf{W}^{[l]\top}$ | passed to layer $l-1$ | $(m, n^{[l-1]})$ |

A useful habit: **every gradient has the shape of the thing it is the gradient of.** $d\mathbf{W}$ has the shape of $\mathbf{W}$, $d\mathbf{A}^{[l-1]}$ has the shape of $\mathbf{A}^{[l-1]}$.

### 5.1 The linear part
`linear_backward` takes $d\mathbf{Z}^{[l]}$ and the linear cache, and returns the last three equations. `dZ.sum(dim=0, keepdim=True)` sums over the rows and keeps the shape $(1, n^{[l]})$; it is the same as $\mathbf{1}^\top d\mathbf{Z}^{[l]}$ in the lecture.

In [10]:
def linear_backward(dZ, cache):
    A_prev, W, b = cache
    m = A_prev.shape[0]                              # examples are rows
    dW = A_prev.T @ dZ / m                           # (n_prev, n_l)
    db = dZ.sum(dim=0, keepdim=True) / m             # (1, n_l)
    dA_prev = dZ @ W.T                               # (m, n_prev)
    return dA_prev, dW, db

torch.manual_seed(2)
dZ = torch.randn(3, 2)                         # same shape as Z in Part 3.1
dA_prev, dW, db = linear_backward(dZ, (A_prev, W, b))
print("dA_prev", tuple(dA_prev.shape), "same as A_prev", tuple(A_prev.shape))
print("dW     ", tuple(dW.shape), "same as W     ", tuple(W.shape))
print("db     ", tuple(db.shape), "same as b     ", tuple(b.shape))

dA_prev (3, 4) same as A_prev (3, 4)
dW      (4, 2) same as W      (4, 2)
db      (1, 2) same as b      (1, 2)


### 5.2 The activation part
- ReLU: $g'(z) = 1$ if $z > 0$, else $0$. So $d\mathbf{Z} = d\mathbf{A} \odot [\mathbf{Z} > 0]$.
- Sigmoid: $g'(z) = \sigma(z)\big(1 - \sigma(z)\big)$.

`linear_activation_backward` first turns $d\mathbf{A}^{[l]}$ into $d\mathbf{Z}^{[l]}$, then calls `linear_backward`.

In [11]:
def relu_backward(dA, cache):
    Z = cache
    return dA * (Z > 0).float()                      # g'(Z) = 1 where Z > 0, else 0

def sigmoid_backward(dA, cache):
    Z = cache
    s = torch.sigmoid(Z)
    return dA * s * (1 - s)                          # g'(Z) = s (1 - s)

def linear_activation_backward(dA, cache, activation):
    linear_cache, activation_cache = cache
    if activation == "relu":
        dZ = relu_backward(dA, activation_cache)
    elif activation == "sigmoid":
        dZ = sigmoid_backward(dA, activation_cache)
    return linear_backward(dZ, linear_cache)

_, cache_relu = linear_activation_forward(A_prev, W, b, "relu")
dA = torch.ones(3, 2)
print("Z =\n", cache_relu[1])
print("dZ for ReLU, with dA = 1 everywhere:\n", relu_backward(dA, cache_relu[1]))

Z =
 tensor([[ 0.2210, -0.5107],
        [ 1.9513, -2.9794],
        [ 1.7237, -1.8384]])
dZ for ReLU, with dA = 1 everywhere:
 tensor([[1., 0.],
        [1., 0.],
        [1., 0.]])


With $d\mathbf{A} = 1$ everywhere, $d\mathbf{Z}$ shows the ReLU derivative itself: 1 where $\mathbf{Z} > 0$, 0 where $\mathbf{Z} \le 0$.

### 5.3 The whole backward pass, and where it starts
The backward pass needs a starting point: $d\mathbf{A}^{[L]}$, the derivative of the loss with respect to the output. For cross-entropy,
$$d\mathbf{A}^{[L]} = -\left(\frac{\mathbf{Y}}{\mathbf{A}^{[L]}} - \frac{1 - \mathbf{Y}}{1 - \mathbf{A}^{[L]}}\right) \qquad \text{(elementwise)}$$
Through the sigmoid, $d\mathbf{Z}^{[L]} = d\mathbf{A}^{[L]} \odot \mathbf{A}^{[L]}(1 - \mathbf{A}^{[L]})$, and everything simplifies to the result of the lecture:
$$d\mathbf{Z}^{[L]} = \mathbf{A}^{[L]} - \mathbf{Y}$$
Let us check that both routes give the same $d\mathbf{Z}^{[L]}$:

In [12]:
Y_small = torch.tensor([[1.0], [0.0], [1.0], [1.0], [0.0], [0.0]])
_, Z_L = caches[-1]
dAL = -(Y_small / AL - (1 - Y_small) / (1 - AL))
print("through sigmoid_backward:", sigmoid_backward(dAL, Z_L).flatten())
print("AL - Y:                  ", (AL - Y_small).flatten())

through sigmoid_backward: tensor([-0.3942,  0.5583, -0.5485, -0.4621,  0.4829,  0.5692])
AL - Y:                   tensor([-0.3942,  0.5583, -0.5485, -0.4621,  0.4829,  0.5692])


Same numbers. So why not always start from $d\mathbf{A}^{[L]}$? Because it **divides** by $\mathbf{A}^{[L]}$ and by $1 - \mathbf{A}^{[L]}$. When the network becomes very confident, the sigmoid rounds to exactly 1 in float32, and the division breaks:

In [13]:
a = torch.sigmoid(torch.tensor(20.0))      # a confident output
y = torch.tensor(1.0)
print("a =", a.item(), "| 1 - a =", (1 - a).item())
print("dA through the division:", (-(y / a - (1 - y) / (1 - a))).item())
print("dZ = a - y:            ", (a - y).item())

a = 1.0 | 1 - a = 0.0
dA through the division: nan
dZ = a - y:             0.0


$0/0$ gives `nan`, and one `nan` spreads to every parameter at the next update. The shortcut $\mathbf{A}^{[L]} - \mathbf{Y}$ never divides, so `L_model_backward` uses it for the output layer, then calls `linear_activation_backward` with ReLU for layers $L-1, \dots, 1$. (PyTorch does the same thing inside `nn.BCEWithLogitsLoss`.)

In [14]:
def L_model_backward(AL, Y, caches):
    grads = {}
    L = len(caches)
    # Output layer: sigmoid + cross-entropy together give dZ = AL - Y (Lecture 2).
    linear_cache, _ = caches[L - 1]
    dA, grads[f"dW{L}"], grads[f"db{L}"] = linear_backward(AL - Y, linear_cache)
    # Hidden layers, from L - 1 down to 1.
    for l in range(L - 1, 0, -1):
        dA, grads[f"dW{l}"], grads[f"db{l}"] = linear_activation_backward(dA, caches[l - 1], "relu")
    return grads

grads = L_model_backward(AL, Y_small, caches)
for k, g in grads.items():
    print(f"{k}: {tuple(g.shape)}")

dW3: (3, 1)
db3: (1, 1)
dW2: (5, 3)
db2: (1, 3)
dW1: (4, 5)
db1: (1, 5)


### 5.4 Checking with autograd
Shapes are right, but are the values right? Autograd computes the same gradients automatically from the forward pass. The next function compares our gradients with autograd's, parameter by parameter. It takes the forward and backward functions as arguments, so you can reuse it in the exercises.

In [15]:
def compare_with_autograd(forward, backward, params, X, Y):
    p = {k: v.clone().requires_grad_() for k, v in params.items()}
    AL, _ = forward(X, p)
    compute_cost(AL, Y).backward()                      # autograd
    AL, caches = forward(X, params)
    grads = backward(AL, Y, caches)                     # our code
    for k in params:
        g = grads["d" + k]
        print(f"d{k:3s} shape {str(tuple(g.shape)):7s} | max difference from autograd {(g - p[k].grad).abs().max().item():.1e}")

compare_with_autograd(L_model_forward, L_model_backward, params_small, X_small, Y_small)

dW1  shape (4, 5)  | max difference from autograd 1.5e-08
db1  shape (1, 5)  | max difference from autograd 7.5e-09
dW2  shape (5, 3)  | max difference from autograd 1.5e-08
db2  shape (1, 3)  | max difference from autograd 3.7e-09
dW3  shape (3, 1)  | max difference from autograd 7.5e-09
db3  shape (1, 1)  | max difference from autograd 3.7e-09


All six differences are of the order of $10^{-8}$, which is float32 rounding: our backward pass is correct. This check replaces the hidden test files of other courses: **whenever you write a backward pass, compare it with autograd.**

## 6 · Updating the parameters
$$\mathbf{W}^{[l]} \leftarrow \mathbf{W}^{[l]} - \alpha\, d\mathbf{W}^{[l]}, \qquad \mathbf{b}^{[l]} \leftarrow \mathbf{b}^{[l]} - \alpha\, d\mathbf{b}^{[l]}$$
`update_parameters` returns a **new** dictionary, so the old parameters are not changed by accident.

In [16]:
def update_parameters(params, grads, lr):
    # One step of gradient descent: p <- p - lr * dp, for every W and b.
    return {k: params[k] - lr * grads["d" + k] for k in params}

new_params = update_parameters(params_small, grads, lr=0.1)
print("b1 before:", params_small["b1"])
print("b1 after: ", new_params["b1"])
print("check:    ", params_small["b1"] - 0.1 * grads["db1"])

b1 before: tensor([[0., 0., 0., 0., 0.]])
b1 after:  tensor([[-0.0075,  0.0012,  0.0033,  0.0099, -0.0057]])
check:     tensor([[-0.0075,  0.0012,  0.0033,  0.0099, -0.0057]])


## 7 · One step of gradient descent

Every piece is ready. One step of training is: forward, cost, backward, update. Below, we take one step on the 6 small examples and compute the cost again with the new parameters.

In [17]:
AL, caches = L_model_forward(X_small, params_small)
cost_before = compute_cost(AL, Y_small).item()
grads = L_model_backward(AL, Y_small, caches)
params_after = update_parameters(params_small, grads, lr=0.5)
cost_after = compute_cost(L_model_forward(X_small, params_after)[0], Y_small).item()
print(f"cost before the step: {cost_before:.4f}")
print(f"cost after the step:  {cost_after:.4f}")

cost before the step: 0.7059
cost after the step:  0.5691


---
## ★ Exercises (on your own)

### Exercise 1 · Shapes and number of parameters
For `layer_dims = [784, 64, 32, 1]` and a batch of $m = 10000$ examples:

**(a)** For every layer, print the shapes of $\mathbf{W}^{[l]}$, $\mathbf{b}^{[l]}$ and $\mathbf{Z}^{[l]}$, **without** running the network: compute them from `layer_dims` and `m`.

**(b)** Print the total number of parameters (all the entries of all the $\mathbf{W}$ and $\mathbf{b}$), then check it by counting the entries of `initialize_parameters_deep(layer_dims)`.

**(c)** Do the same count for `[784, 20, 7, 5, 1]`. Which network has more parameters, and roughly how many times more?

Expected output:
```
layer 1: W (784, 64)  b (1, 64)  Z (10000, 64)
layer 2: W (64, 32)   b (1, 32)  Z (10000, 32)
layer 3: W (32, 1)    b (1, 1)   Z (10000, 1)
total parameters: 52353 | counted: 52353
[784, 20, 7, 5, 1]: 15893 | ratio 3.3
```

In [18]:
dims = [784, 64, 32, 1]
m = 10000

# (a) shapes computed from layer_dims and m only (no network is run)
for l in range(1, len(dims)):
    print(f"layer {l}: W {str((dims[l-1], dims[l])):10s} b {str((1, dims[l])):8s} Z {(m, dims[l])}")

# (b) total number of parameters: computed, then counted
def n_params(d):
    return sum(d[l-1] * d[l] + d[l] for l in range(1, len(d)))   # W: n_prev*n_l, b: n_l

counted = sum(p.numel() for p in initialize_parameters_deep(dims).values())
print(f"total parameters: {n_params(dims)} | counted: {counted}")

# (c) the same for [784, 20, 7, 5, 1], and the ratio
dims2 = [784, 20, 7, 5, 1]
print(f"{dims2}: {n_params(dims2)} | ratio {n_params(dims) / n_params(dims2):.1f}")


layer 1: W (784, 64)  b (1, 64)  Z (10000, 64)
layer 2: W (64, 32)   b (1, 32)  Z (10000, 32)
layer 3: W (32, 1)    b (1, 1)   Z (10000, 1)
total parameters: 52353 | counted: 52353
[784, 20, 7, 5, 1]: 15893 | ratio 3.3


**Exercise 1(c) · answer.** `[784, 64, 32, 1]` has more parameters: 52,353 against 15,893, about **3.3 times more**. Most of them are in the first layer, whose $\mathbf{W}^{[1]}$ has $784 \times 64 = 50{,}176$ entries; so the width of the first hidden layer, not the depth, dominates the count.

### Exercise 2 · Leaky ReLU
Leaky ReLU is $g(z) = z$ if $z > 0$, else $0.01\,z$. Its derivative is 1 if $z > 0$, else $0.01$. Unlike ReLU, it never has a zero gradient.

**(a)** Write `leaky_relu(Z)` (returns `(A, Z)`) and `leaky_relu_backward(dA, cache)`.

**(b)** Write `L_model_forward_leaky(X, params)` and `L_model_backward_leaky(AL, Y, caches)`: copies of `L_model_forward` and `L_model_backward` with leaky ReLU in the hidden layers. *Hint:* the simplest way is to add a `"leaky_relu"` case to copies of `linear_activation_forward` and `linear_activation_backward`.

**(c)** Check your gradients with `compare_with_autograd(L_model_forward_leaky, L_model_backward_leaky, params_small, X_small, Y_small)`. Every difference should be about $10^{-7}$ or smaller; your tiny numbers may not match the expected output exactly.

Expected output of (c):
```
dW1  shape (4, 5)  | max difference from autograd 3.0e-08
db1  shape (1, 5)  | max difference from autograd 1.5e-08
dW2  shape (5, 3)  | max difference from autograd 1.5e-08
db2  shape (1, 3)  | max difference from autograd 1.5e-08
dW3  shape (3, 1)  | max difference from autograd 4.5e-08
db3  shape (1, 1)  | max difference from autograd 1.1e-08
```

In [19]:
def leaky_relu(Z, slope=0.01):
    return torch.where(Z > 0, Z, slope * Z), Z          # (A, cache); the cache is Z

def leaky_relu_backward(dA, cache, slope=0.01):
    Z = cache
    return dA * torch.where(Z > 0, torch.ones_like(Z), torch.full_like(Z, slope))   # g'(Z) = 1 or 0.01

# (b) copies of the linear_activation functions with a "leaky_relu" case
def linear_activation_forward_leaky(A_prev, W, b, activation):
    Z, linear_cache = linear_forward(A_prev, W, b)
    if activation == "sigmoid":
        A, activation_cache = sigmoid(Z)
    elif activation == "relu":
        A, activation_cache = relu(Z)
    elif activation == "leaky_relu":
        A, activation_cache = leaky_relu(Z)
    return A, (linear_cache, activation_cache)

def linear_activation_backward_leaky(dA, cache, activation):
    linear_cache, activation_cache = cache
    if activation == "relu":
        dZ = relu_backward(dA, activation_cache)
    elif activation == "sigmoid":
        dZ = sigmoid_backward(dA, activation_cache)
    elif activation == "leaky_relu":
        dZ = leaky_relu_backward(dA, activation_cache)
    return linear_backward(dZ, linear_cache)

def L_model_forward_leaky(X, params):
    # [LINEAR -> LEAKY_RELU] x (L - 1), then LINEAR -> SIGMOID.
    caches = []
    A = X
    L = len(params) // 2
    for l in range(1, L):
        A, cache = linear_activation_forward_leaky(A, params[f"W{l}"], params[f"b{l}"], "leaky_relu")
        caches.append(cache)
    AL, cache = linear_activation_forward_leaky(A, params[f"W{L}"], params[f"b{L}"], "sigmoid")
    caches.append(cache)
    return AL, caches

def L_model_backward_leaky(AL, Y, caches):
    grads = {}
    L = len(caches)
    linear_cache, _ = caches[L - 1]
    dA, grads[f"dW{L}"], grads[f"db{L}"] = linear_backward(AL - Y, linear_cache)   # dZ[L] = AL - Y
    for l in range(L - 1, 0, -1):
        dA, grads[f"dW{l}"], grads[f"db{l}"] = linear_activation_backward_leaky(dA, caches[l - 1], "leaky_relu")
    return grads

# (c) compare with autograd
compare_with_autograd(L_model_forward_leaky, L_model_backward_leaky, params_small, X_small, Y_small)


dW1  shape (4, 5)  | max difference from autograd 3.0e-08
db1  shape (1, 5)  | max difference from autograd 1.5e-08
dW2  shape (5, 3)  | max difference from autograd 1.5e-08
db2  shape (1, 3)  | max difference from autograd 1.5e-08
dW3  shape (3, 1)  | max difference from autograd 4.5e-08
db3  shape (1, 1)  | max difference from autograd 1.1e-08


### Exercise 3 · Find the bugs
The function below contains **two bugs**. Neither of them raises an error: the shapes are all correct, the gradients are just wrong.

**(a)** Run `compare_with_autograd(L_model_forward, L_model_backward_buggy, params_small, X_small, Y_small)`. Which gradients are wrong?

**(b)** Find the two bugs, write a fixed copy `L_model_backward_fixed`, and run the comparison again. All differences should be about $10^{-8}$.

**(c)** In a markdown cell, explain each bug in one sentence. One of them is a typical mistake when you copy code written for the **column** convention.

Expected output of (b):
```
dW1  shape (4, 5)  | max difference from autograd 1.5e-08
db1  shape (1, 5)  | max difference from autograd 7.5e-09
dW2  shape (5, 3)  | max difference from autograd 1.5e-08
db2  shape (1, 3)  | max difference from autograd 0.0e+00
dW3  shape (3, 1)  | max difference from autograd 7.5e-09
db3  shape (1, 1)  | max difference from autograd 3.7e-09
```

In [ ]:
# (a) The buggy function gives differences of about 1 for all six gradients, so every gradient is wrong.
def L_model_backward_buggy(AL, Y, caches):
    grads = {}
    L = len(caches)
    m = AL.shape[1]
    dA = -(Y / AL - (1 - Y) / (1 - AL))
    for l in range(L, 0, -1):
        (A_prev, W, b), Z = caches[l - 1]
        if l == L:
            dZ = dA * torch.sigmoid(Z)
        else:
            dZ = dA * (Z > 0).float()
        grads[f"dW{l}"] = A_prev.T @ dZ / m
        grads[f"db{l}"] = dZ.sum(dim=0, keepdim=True) / m
        dA = dZ @ W.T
    return grads

compare_with_autograd(L_model_forward, L_model_backward_buggy, params_small, X_small, Y_small)

dW1  shape (4, 5)  | max difference from autograd 1.7e+00
db1  shape (1, 5)  | max difference from autograd 1.0e+00
dW2  shape (5, 3)  | max difference from autograd 1.8e+00
db2  shape (1, 3)  | max difference from autograd 1.1e+00
dW3  shape (3, 1)  | max difference from autograd 8.2e-01
db3  shape (1, 1)  | max difference from autograd 4.8e-01


In [ ]:
# (b) the fixed copy: two changes, marked FIX 1 and FIX 2
def L_model_backward_fixed(AL, Y, caches):
    grads = {}
    L = len(caches)
    m = AL.shape[0]                                    # FIX 1: examples are rows, so m = shape[0]
    dA = -(Y / AL - (1 - Y) / (1 - AL))
    for l in range(L, 0, -1):
        (A_prev, W, b), Z = caches[l - 1]
        if l == L:
            s = torch.sigmoid(Z)
            dZ = dA * s * (1 - s)                      # FIX 2: g'(Z) = s (1 - s), not just s
        else:
            dZ = dA * (Z > 0).float()
        grads[f"dW{l}"] = A_prev.T @ dZ / m
        grads[f"db{l}"] = dZ.sum(dim=0, keepdim=True) / m
        dA = dZ @ W.T
    return grads

compare_with_autograd(L_model_forward, L_model_backward_fixed, params_small, X_small, Y_small)


dW1  shape (4, 5)  | max difference from autograd 1.5e-08
db1  shape (1, 5)  | max difference from autograd 7.5e-09
dW2  shape (5, 3)  | max difference from autograd 1.5e-08
db2  shape (1, 3)  | max difference from autograd 0.0e+00
dW3  shape (3, 1)  | max difference from autograd 7.5e-09
db3  shape (1, 1)  | max difference from autograd 3.7e-09


**Exercise 3 · answers**

**(a)** All six gradients (`dW1`, `db1`, ..., `dW3`, `db3`) are wrong, not only those of one layer: the first bug is in the output layer's `dZ`, and every earlier gradient is computed from it by the chain rule, so the error is passed to every layer (the second bug also rescales every gradient).

**(c)**
- **Bug 1: `m = AL.shape[1]`.** This is the column convention (examples in columns, so `m = AL.shape[1]`); here examples are rows and `AL` has shape `(m, 1)`, so `shape[1]` is 1 and every gradient is divided by 1 instead of by `m` (it is `m` times too large). The fix is `m = AL.shape[0]`.
- **Bug 2: `dZ = dA * torch.sigmoid(Z)`.** The derivative of the sigmoid is `s * (1 - s)`, not `s`; the factor `(1 - s)` is missing, so `dZ` of the output layer is wrong. The fix is `dZ = dA * s * (1 - s)`.

*Remark:* the fixed function still divides by `AL` and `1 - AL`, so it breaks when the network becomes very confident (Part 5.3); `dZ = AL - Y` is the safe way to start.